# 🌳 Clase 6 — Árboles de Decisión y Random Forest

### Aplicaciones de Minería de Datos a Economía y Finanzas
**Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026**

*Dr. Darío Ezequiel Díaz · Jueves 14 de mayo de 2026*

---

## 📋 Objetivos de la práctica

Al finalizar este notebook, ustedes serán capaces de:

1. **Entrenar** un árbol de decisión CART sobre el German Credit Dataset, controlando la complejidad mediante hiperparámetros de pre-poda.
2. **Diagnosticar** el sobreajuste arbóreo trazando las curvas de error de entrenamiento y validación en función de la profundidad.
3. **Aplicar** post-poda por *cost-complexity pruning* y seleccionar el `ccp_alpha` óptimo mediante validación cruzada.
4. **Construir** un Random Forest con doble aleatorización (bootstrap + subsampleo de variables) y monitorear su convergencia mediante el error *out-of-bag*.
5. **Computar** la importancia de variables por dos enfoques alternativos: *Mean Decrease in Impurity* (MDI) y *permutation importance*, identificando sus respectivos sesgos.
6. **Comparar** el rendimiento de CART y Random Forest contra la regresión logística de la Clase 5, mediante AUC, KS y Gini.

## 🗺️ Mapa del notebook

| Sección | Contenido |
|---------|-----------|
| 1 | Configuración del entorno |
| 2 | Carga del dataset German Credit |
| 3 | Preprocesamiento: encoding y partición |
| 4 | Árbol de decisión CART |
| 5 | Análisis del sobreajuste y poda |
| 6 | Random Forest: entrenamiento y diagnóstico |
| 7 | Importancia de variables: MDI vs *permutation* |
| 8 | Comparativa final con regresión logística |
| 9 | Síntesis y entregable |

---

> 💡 **Recomendación:** ejecuten las celdas en orden secuencial. Cada sección depende de los objetos definidos en las anteriores. La semilla del curso es `RNG_SEED = 2026` y se respeta en todos los modelos para garantizar reproducibilidad.


## 1. Configuración del entorno

Instalamos las librerías necesarias y fijamos la semilla aleatoria del curso (`RNG_SEED = 2026`). Trabajaremos con `scikit-learn` como librería principal: provee los estimadores `DecisionTreeClassifier` y `RandomForestClassifier`, las funciones de inspección, y las métricas que vamos a calcular.

### Librerías utilizadas

- **`pandas`, `numpy`**: manipulación de datos y cálculo numérico
- **`matplotlib`, `seaborn`**: visualización
- **`scikit-learn`**: estimadores arbóreos, métricas, partición y validación
- **`scipy`**: cálculo del estadístico KS

In [ ]:
# Instalación de librerías (silenciosa si ya están disponibles)
import subprocess
import sys

paquetes_requeridos = ['scikit-learn', 'pandas', 'numpy',
                       'matplotlib', 'seaborn', 'scipy']

for pkg in paquetes_requeridos:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        print(f"📦 Instalando {pkg}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                              '--quiet', '--break-system-packages', pkg])

print("✅ Todas las librerías están disponibles")

In [ ]:
# Imports principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             classification_report, accuracy_score)
from scipy.stats import ks_2samp

# Configuración visual — paleta Unidad 3 (Royal Indigo & Amber)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 100
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

COL_NAVY    = '#1A2148'
COL_VIOLET  = '#6E3FD0'
COL_AMBER   = '#D97706'
COL_EMERALD = '#14B8A6'
COL_CARMIN  = '#DC2626'
COL_GOLD    = '#F5C443'

# Semilla del curso
RNG_SEED = 2026
np.random.seed(RNG_SEED)
warnings.filterwarnings('ignore')

print(f"🌳 Clase 6 — entorno configurado")
print(f"   Semilla aleatoria: RNG_SEED = {RNG_SEED}")
print(f"   scikit-learn = {__import__('sklearn').__version__}")

## 2. Carga del German Credit Dataset

Cargamos el conjunto de datos desde **OpenML** mediante `sklearn.datasets.fetch_openml('credit-g')`. Este dataset, originado en la investigación de Hofmann (Universidad de Hamburgo, 1994), contiene **1.000 solicitudes de crédito** con 20 variables explicativas y una variable respuesta binaria:

- `class = 'good'`: el solicitante pagó regularmente (700 casos, 70 %)
- `class = 'bad'`: el solicitante incurrió en default (300 casos, 30 %)

Recodificamos la variable respuesta siguiendo la convención regulatoria: **`Y = 1` representa default** (el evento de interés), `Y = 0` representa pago regular. Esta inversión respecto del valor literal `'bad'` es la práctica habitual en *credit scoring*: la clase positiva es la que el modelo busca detectar.

In [ ]:
def cargar_german_credit():
    '''
    Carga el German Credit Dataset con estrategia de fallback en tres niveles.
    Retorna un DataFrame con las 20 covariables y la variable respuesta 'class'.
    '''
    # Intento 1: OpenML
    try:
        print("⏳ Intentando cargar desde OpenML...")
        data = fetch_openml('credit-g', version=1, as_frame=True, parser='auto')
        df = data.frame
        print(f"✅ Dataset cargado desde OpenML: {df.shape[0]} filas × {df.shape[1]} columnas")
        return df
    except Exception as e:
        print(f"⚠️  OpenML no disponible: {str(e)[:80]}")

    # Intento 2: Google Drive (en Colab)
    try:
        ruta_drive = '/content/drive/MyDrive/datasets/german_credit.csv'
        df = pd.read_csv(ruta_drive)
        print(f"✅ Dataset cargado desde Drive: {df.shape[0]} × {df.shape[1]}")
        return df
    except Exception:
        pass

    # Intento 3: URL pública del repositorio
    try:
        url = 'https://www.openml.org/data/get_csv/31/dataset_31_credit-g.arff'
        df = pd.read_csv(url)
        print(f"✅ Dataset cargado desde URL pública: {df.shape[0]} × {df.shape[1]}")
        return df
    except Exception as e:
        raise RuntimeError(f"❌ No se pudo cargar el dataset por ninguna vía. Último error: {e}")


df = cargar_german_credit()
df.head()

In [ ]:
# Recodificación de la variable respuesta según convención regulatoria
# Y = 1 → default (clase positiva, evento de interés)
# Y = 0 → pago regular

df['Y'] = (df['class'] == 'bad').astype(int)
df = df.drop(columns=['class'])

print(f"📊 Distribución de la variable respuesta:")
print(f"   Y = 0 (pago regular): {(df['Y']==0).sum():>4} casos ({(df['Y']==0).mean():.1%})")
print(f"   Y = 1 (default):      {(df['Y']==1).sum():>4} casos ({(df['Y']==1).mean():.1%})")
print(f"\n   Ratio buenos:malos = {(df['Y']==0).sum()/(df['Y']==1).sum():.2f}:1")

## 3. Preprocesamiento

Los árboles de decisión y los Random Forest tienen una ventaja operativa importante respecto de la regresión logística: **no requieren estandarización ni escalado de variables**. Las particiones recursivas son invariantes ante transformaciones monótonas. Tampoco asumen distribuciones particulares ni linealidad.

Sin embargo, la implementación de `scikit-learn` no acepta directamente variables categóricas: requiere codificarlas numéricamente. Aplicamos **one-hot encoding** sin descartar la primera categoría (`drop_first=False`), ya que los árboles no sufren multicolinealidad como la logística.

Posteriormente realizamos la **partición train/test estratificada** preservando la proporción de defaults en ambos conjuntos. La semilla `RNG_SEED = 2026` garantiza reproducibilidad.

In [ ]:
# Identificar variables por tipo
cat_cols = df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
num_cols = df.select_dtypes(include=['number']).columns.tolist()
num_cols = [c for c in num_cols if c != 'Y']

print(f"📦 Estructura del dataset:")
print(f"   Variables categóricas: {len(cat_cols)}")
print(f"   Variables numéricas:   {len(num_cols)}")
print(f"   Variable objetivo:     Y (binaria)")
print(f"\n   Categóricas: {cat_cols[:5]}{'...' if len(cat_cols)>5 else ''}")
print(f"   Numéricas:   {num_cols}")

In [ ]:
# One-hot encoding de las variables categóricas
# Para árboles, mantenemos drop_first=False — no hay riesgo de multicolinealidad
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=False)

# Conversión explícita a int para evitar warnings de tipo bool con sklearn
for c in df_encoded.columns:
    if df_encoded[c].dtype == bool:
        df_encoded[c] = df_encoded[c].astype(int)

print(f"📐 Shape original:    {df.shape}")
print(f"📐 Shape codificado:  {df_encoded.shape}")
print(f"📐 Variables nuevas:  {df_encoded.shape[1] - df.shape[1]} dummies generadas")

In [ ]:
# Partición train/test estratificada por Y
X = df_encoded.drop(columns=['Y'])
y = df_encoded['Y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RNG_SEED
)

print(f"📊 Partición train/test estratificada:")
print(f"   Train: {X_train.shape[0]:>4} obs × {X_train.shape[1]} features  "
      f"(default = {y_train.mean():.1%})")
print(f"   Test:  {X_test.shape[0]:>4} obs × {X_test.shape[1]} features  "
      f"(default = {y_test.mean():.1%})")
print(f"\n✅ Proporción de default preservada en ambos splits")

## 4. Árbol de decisión CART

Entrenamos un primer árbol con hiperparámetros razonables como referencia inicial:

- `criterion='gini'`: criterio de impureza por defecto, más rápido que `entropy` y con resultados muy similares.
- `max_depth=5`: profundidad máxima moderada, evita sobreajuste catastrófico.
- `min_samples_leaf=20`: cada hoja debe contener al menos 20 observaciones — soporte estadístico suficiente.
- `random_state=RNG_SEED`: reproducibilidad ante empates en ganancia.

Este árbol nos servirá como **modelo de referencia exploratorio**. En la sección siguiente afinaremos la complejidad mediante poda.

In [ ]:
# Entrenamiento del árbol CART inicial
cart_inicial = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    min_samples_leaf=20,
    random_state=RNG_SEED
)
cart_inicial.fit(X_train, y_train)

# Predicciones sobre train y test
p_train = cart_inicial.predict_proba(X_train)[:, 1]
p_test  = cart_inicial.predict_proba(X_test)[:, 1]

auc_train = roc_auc_score(y_train, p_train)
auc_test  = roc_auc_score(y_test, p_test)

print(f"🌳 Árbol CART inicial (max_depth=5, min_samples_leaf=20)")
print(f"   Número de nodos:   {cart_inicial.tree_.node_count}")
print(f"   Número de hojas:   {cart_inicial.get_n_leaves()}")
print(f"   Profundidad real:  {cart_inicial.get_depth()}")
print(f"\n📈 Performance:")
print(f"   AUC train: {auc_train:.4f}")
print(f"   AUC test:  {auc_test:.4f}")
print(f"   Brecha:    {auc_train - auc_test:.4f}  (indicador de sobreajuste)")

### 4.1. Visualización del árbol

`scikit-learn` provee la función `plot_tree` para visualizar el árbol completo. Cada nodo muestra:

- **La regla de partición** aplicada (variable y umbral).
- **El valor de Gini** del nodo.
- **El número de observaciones** que llegan al nodo.
- **La distribución de clases** en el nodo (`value=[buenos, malos]`).
- **La clase predicha** (la mayoritaria).

El color de cada nodo refleja la pureza: los nodos azules son mayoritariamente `Y=0` (buenos pagadores) y los anaranjados son mayoritariamente `Y=1` (default). Cuanto más intenso el color, más puro el nodo.

In [ ]:
# Visualización del árbol con plot_tree
fig, ax = plt.subplots(1, 1, figsize=(20, 10))
plot_tree(
    cart_inicial,
    feature_names=X_train.columns.tolist(),
    class_names=['OK', 'Default'],
    filled=True,
    rounded=True,
    fontsize=8,
    impurity=True,
    proportion=False,
    ax=ax
)
ax.set_title('Árbol CART — German Credit (max_depth=5)',
             color=COL_NAVY, fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

### 4.2. Extracción de reglas en formato textual

Una de las virtudes distintivas de los árboles es que pueden traducirse a **reglas en lenguaje natural**. La función `export_text` produce esta representación textual, donde cada trayectoria desde la raíz hasta una hoja define una regla de decisión conjuntiva.

Estas reglas son auditables individualmente: un oficial de crédito puede leerlas, un cliente que recibió rechazo puede entender el porqué, y un regulador puede verificar la lógica. Esta transparencia es la razón por la cual los árboles han persistido durante décadas en la práctica bancaria pese a tener performance inferior a modelos más modernos.

In [ ]:
# Exportar reglas en formato textual (limitamos profundidad para legibilidad)
reglas = export_text(
    cart_inicial,
    feature_names=X_train.columns.tolist(),
    max_depth=3,
    show_weights=False
)
print("📜 Reglas extraídas (primeros 3 niveles):\n")
print(reglas)

## 5. Análisis del sobreajuste y poda

Vamos a diagnosticar empíricamente el sobreajuste arbóreo trazando la **curva en U** característica: error de entrenamiento descendente, error de validación en forma de U con un mínimo intermedio. Para esto entrenamos árboles con profundidades crecientes y registramos AUC en ambos conjuntos.

Posteriormente aplicamos **post-poda por *cost-complexity***, formulada por Breiman et al. (1984):

$$R_\alpha(T) = R(T) + \alpha \cdot |\widetilde{T}|$$

donde $R(T)$ es el error del árbol sobre el entrenamiento, $|\widetilde{T}|$ es el número de hojas, y $\alpha \geq 0$ es el parámetro de penalización. Para cada valor de $\alpha$ existe un sub-árbol óptimo, y la secuencia es única y anidada (Teorema de Breiman).

In [ ]:
# Curva de sobreajuste: AUC train vs AUC test en función de max_depth
profundidades = list(range(1, 16))
resultados_curva = []

for d in profundidades:
    m = DecisionTreeClassifier(max_depth=d, random_state=RNG_SEED)
    m.fit(X_train, y_train)
    auc_tr = roc_auc_score(y_train, m.predict_proba(X_train)[:, 1])
    auc_ts = roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])
    resultados_curva.append({'profundidad': d, 'auc_train': auc_tr,
                             'auc_test': auc_ts, 'gap': auc_tr - auc_ts,
                             'n_hojas': m.get_n_leaves()})

curva_df = pd.DataFrame(resultados_curva)
print(curva_df.round(4).to_string(index=False))

In [ ]:
# Visualización de la curva de sobreajuste
fig, ax = plt.subplots(1, 1, figsize=(11, 5.5))

ax.plot(curva_df['profundidad'], curva_df['auc_train'],
        marker='o', linewidth=2.5, color=COL_VIOLET,
        label='AUC entrenamiento')
ax.plot(curva_df['profundidad'], curva_df['auc_test'],
        marker='s', linewidth=2.5, color=COL_AMBER,
        label='AUC validación (test)')

# Marcar el óptimo
idx_optimo = curva_df['auc_test'].idxmax()
prof_optima = curva_df.loc[idx_optimo, 'profundidad']
auc_optimo  = curva_df.loc[idx_optimo, 'auc_test']
ax.axvline(prof_optima, color=COL_CARMIN, linestyle='--', linewidth=1.5, alpha=0.7)
ax.scatter([prof_optima], [auc_optimo], color=COL_CARMIN, s=200, zorder=5,
           edgecolor='white', linewidth=2, label=f'Óptimo: depth={prof_optima}')

ax.set_xlabel('Profundidad máxima del árbol', fontsize=12, color=COL_NAVY)
ax.set_ylabel('AUC-ROC', fontsize=12, color=COL_NAVY)
ax.set_title('Diagnóstico de sobreajuste: AUC train vs test por profundidad',
             fontsize=13, fontweight='bold', color=COL_NAVY)
ax.legend(loc='lower right', frameon=True, fontsize=11)
ax.grid(alpha=0.4)
ax.set_xticks(profundidades)

plt.tight_layout()
plt.show()

print(f"\n🎯 Profundidad óptima por AUC test: {prof_optima} (AUC={auc_optimo:.4f})")
print(f"   Brecha train-test en óptimo: {curva_df.loc[idx_optimo, 'gap']:.4f}")

### 5.1. Cost-complexity pruning: secuencia de sub-árboles

`scikit-learn` implementa el algoritmo de *weakest link pruning* mediante el método `cost_complexity_pruning_path`. Este método retorna los valores críticos de $\alpha$ junto con las impurezas correspondientes — los puntos donde el algoritmo poda una rama.

A continuación entrenamos un árbol sin restricciones (`max_depth=None`) y obtenemos su secuencia de poda completa.

In [ ]:
# Árbol completo sin podar como punto de partida
cart_pleno = DecisionTreeClassifier(random_state=RNG_SEED)
cart_pleno.fit(X_train, y_train)

print(f"🌲 Árbol pleno (sin restricciones):")
print(f"   Nodos:       {cart_pleno.tree_.node_count}")
print(f"   Hojas:       {cart_pleno.get_n_leaves()}")
print(f"   Profundidad: {cart_pleno.get_depth()}")
print(f"   AUC train:   {roc_auc_score(y_train, cart_pleno.predict_proba(X_train)[:, 1]):.4f}")
print(f"   AUC test:    {roc_auc_score(y_test, cart_pleno.predict_proba(X_test)[:, 1]):.4f}")

# Secuencia de poda
path = cart_pleno.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas
impurezas  = path.impurities

print(f"\n📐 Secuencia de poda generada:")
print(f"   Valores de α: {len(ccp_alphas)} puntos")
print(f"   Rango α:      [{ccp_alphas.min():.5f}, {ccp_alphas.max():.5f}]")

### 5.2. Selección óptima de `ccp_alpha` por validación cruzada

La forma rigurosa de seleccionar $\alpha$ es por **validación cruzada estratificada**. Para cada valor candidato de la secuencia, entrenamos un árbol con esa penalización y medimos el AUC promedio sobre los 5 folds. El $\alpha$ óptimo es el que maximiza este promedio.

In [ ]:
# Selección de ccp_alpha por validación cruzada 5-fold estratificada
# Para eficiencia, muestreamos hasta 30 valores de α
n_alphas_muestra = min(30, len(ccp_alphas))
indices_muestra = np.linspace(0, len(ccp_alphas) - 1, n_alphas_muestra).astype(int)
alphas_candidatos = ccp_alphas[indices_muestra]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG_SEED)
auc_cv_medio = []
auc_cv_std   = []

print("⏳ Evaluando ccp_alpha por CV (puede tardar unos segundos)...")
for alpha in alphas_candidatos:
    if alpha < 0:
        alpha = 0
    m = DecisionTreeClassifier(ccp_alpha=alpha, random_state=RNG_SEED)
    scores = cross_val_score(m, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    auc_cv_medio.append(scores.mean())
    auc_cv_std.append(scores.std())

resultados_cv = pd.DataFrame({
    'ccp_alpha': alphas_candidatos,
    'auc_cv_medio': auc_cv_medio,
    'auc_cv_std':   auc_cv_std
})

idx_alpha_opt = resultados_cv['auc_cv_medio'].idxmax()
alpha_opt = resultados_cv.loc[idx_alpha_opt, 'ccp_alpha']
auc_opt_cv = resultados_cv.loc[idx_alpha_opt, 'auc_cv_medio']

print(f"\n🎯 ccp_alpha óptimo = {alpha_opt:.5f}")
print(f"   AUC en CV: {auc_opt_cv:.4f} ± {resultados_cv.loc[idx_alpha_opt, 'auc_cv_std']:.4f}")

In [ ]:
# Visualización de la curva AUC vs ccp_alpha
fig, ax = plt.subplots(1, 1, figsize=(11, 5.5))

auc_arr  = np.array(auc_cv_medio)
std_arr  = np.array(auc_cv_std)

ax.plot(alphas_candidatos, auc_arr, marker='o', linewidth=2.5,
        color=COL_VIOLET, label='AUC promedio CV')
ax.fill_between(alphas_candidatos, auc_arr - std_arr, auc_arr + std_arr,
                alpha=0.2, color=COL_VIOLET, label='± 1 desv. estándar')

ax.axvline(alpha_opt, color=COL_CARMIN, linestyle='--', linewidth=1.8, alpha=0.7)
ax.scatter([alpha_opt], [auc_opt_cv], color=COL_CARMIN, s=180, zorder=5,
           edgecolor='white', linewidth=2,
           label=f'α óptimo = {alpha_opt:.4f}')

ax.set_xlabel(r'$\alpha$ (cost-complexity)', fontsize=12, color=COL_NAVY)
ax.set_ylabel('AUC promedio (5-fold CV)', fontsize=12, color=COL_NAVY)
ax.set_title('Selección de ccp_alpha por validación cruzada',
             fontsize=13, fontweight='bold', color=COL_NAVY)
ax.legend(loc='lower left', frameon=True, fontsize=11)
ax.grid(alpha=0.4)
ax.set_xscale('symlog', linthresh=1e-4)

plt.tight_layout()
plt.show()

### 5.3. Árbol CART final podado

Entrenamos el árbol final con el `ccp_alpha` óptimo seleccionado, sobre todo el conjunto de entrenamiento. Este será el modelo CART que compararemos posteriormente con el Random Forest y con la regresión logística de Clase 5.

In [ ]:
# Árbol final con ccp_alpha óptimo
cart_final = DecisionTreeClassifier(ccp_alpha=alpha_opt, random_state=RNG_SEED)
cart_final.fit(X_train, y_train)

p_train_cart = cart_final.predict_proba(X_train)[:, 1]
p_test_cart  = cart_final.predict_proba(X_test)[:, 1]

auc_train_cart = roc_auc_score(y_train, p_train_cart)
auc_test_cart  = roc_auc_score(y_test, p_test_cart)

print(f"🌳 CART final podado (ccp_alpha = {alpha_opt:.5f})")
print(f"   Nodos:        {cart_final.tree_.node_count}")
print(f"   Hojas:        {cart_final.get_n_leaves()}")
print(f"   Profundidad:  {cart_final.get_depth()}")
print(f"\n📈 Performance:")
print(f"   AUC train:    {auc_train_cart:.4f}")
print(f"   AUC test:     {auc_test_cart:.4f}")
print(f"   Brecha:       {auc_train_cart - auc_test_cart:.4f}")
print(f"\n   👉 Brecha reducida respecto al árbol inicial: la poda controló el sobreajuste.")

## 6. Random Forest: entrenamiento y diagnóstico

Pasamos del árbol individual al ensamble. Random Forest construye $B$ árboles independientes mediante **doble aleatorización**:

1. **Bootstrap de filas**: cada árbol se entrena sobre una muestra bootstrap del conjunto de entrenamiento (con reemplazo, tamaño $n$). En promedio, el 36.8 % de las observaciones queda fuera de cada muestra — el conjunto *out-of-bag* (OOB).
2. **Subsampleo de variables**: en cada nodo de cada árbol, sólo se considera un subconjunto aleatorio de $m \ll p$ variables (`max_features='sqrt'` por defecto en clasificación).

Esta doble aleatorización **decorrelaciona los árboles**, reduciendo la varianza del ensamble por debajo del piso que el simple bagging alcanzaría. La predicción se obtiene por **voto mayoritario** sobre los $B$ árboles.

In [ ]:
# Random Forest con hiperparámetros recomendados
rf = RandomForestClassifier(
    n_estimators=300,
    max_features='sqrt',
    min_samples_leaf=5,
    class_weight='balanced',
    oob_score=True,
    n_jobs=-1,
    random_state=RNG_SEED
)
rf.fit(X_train, y_train)

p_train_rf = rf.predict_proba(X_train)[:, 1]
p_test_rf  = rf.predict_proba(X_test)[:, 1]

auc_train_rf = roc_auc_score(y_train, p_train_rf)
auc_test_rf  = roc_auc_score(y_test, p_test_rf)
auc_oob = rf.oob_score_  # accuracy OOB

print(f"🌲 Random Forest (n_estimators=300, max_features='sqrt')")
print(f"   Número de árboles:     {rf.n_estimators}")
print(f"   Variables por split:   {int(np.sqrt(X_train.shape[1]))}  (≈ √{X_train.shape[1]})")
print(f"\n📈 Performance:")
print(f"   AUC train:             {auc_train_rf:.4f}")
print(f"   AUC test:              {auc_test_rf:.4f}")
print(f"   Accuracy OOB:          {auc_oob:.4f}")
print(f"\n   👉 OOB acta como validación intrínseca — equivale asintóticamente a k-fold con k=B.")

### 6.1. Convergencia del error OOB

Una pregunta natural: ¿cuántos árboles necesitamos? La respuesta empírica se obtiene trazando el error OOB en función de `n_estimators`. Mientras la curva siga descendiendo, conviene agregar más árboles; cuando se estabiliza, el beneficio marginal es despreciable y agregar más sólo incrementa costo computacional.

A diferencia del boosting, **agregar árboles al Random Forest nunca empeora la generalización** — la varianza converge a un piso $\rho \sigma^2$ pero no diverge. Esto convierte a `n_estimators` en un hiperparámetro "amable": elegirlo por curva de convergencia es suficiente.

In [ ]:
# Convergencia OOB en función de n_estimators
n_trees_grid = [10, 25, 50, 75, 100, 150, 200, 250, 300, 400, 500]
oob_accs = []
auc_tests_grid = []

print("⏳ Evaluando convergencia OOB...")
for n in n_trees_grid:
    m = RandomForestClassifier(
        n_estimators=n, max_features='sqrt', min_samples_leaf=5,
        class_weight='balanced', oob_score=True, n_jobs=-1,
        random_state=RNG_SEED
    )
    m.fit(X_train, y_train)
    oob_accs.append(m.oob_score_)
    auc_tests_grid.append(roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]))

convergencia_df = pd.DataFrame({
    'n_estimators': n_trees_grid,
    'oob_accuracy': oob_accs,
    'auc_test':     auc_tests_grid
})
print(convergencia_df.round(4).to_string(index=False))

In [ ]:
# Visualización de la convergencia OOB
fig, ax = plt.subplots(1, 1, figsize=(11, 5.5))

ax.plot(convergencia_df['n_estimators'], convergencia_df['oob_accuracy'],
        marker='o', linewidth=2.5, color=COL_EMERALD, label='Accuracy OOB')
ax.plot(convergencia_df['n_estimators'], convergencia_df['auc_test'],
        marker='s', linewidth=2.5, color=COL_VIOLET, label='AUC test')

ax.axvline(300, color=COL_AMBER, linestyle='--', linewidth=1.8, alpha=0.7,
           label='n=300 (elegido)')

ax.set_xlabel('Número de árboles (n_estimators)', fontsize=12, color=COL_NAVY)
ax.set_ylabel('Métrica', fontsize=12, color=COL_NAVY)
ax.set_title('Convergencia del Random Forest: OOB vs número de árboles',
             fontsize=13, fontweight='bold', color=COL_NAVY)
ax.legend(loc='lower right', frameon=True, fontsize=11)
ax.grid(alpha=0.4)

plt.tight_layout()
plt.show()

## 7. Importancia de variables: MDI vs Permutation Importance

Los modelos de ensemble pierden la interpretabilidad visual del árbol individual. A cambio ofrecen una medida cuantitativa global de la relevancia de cada predictor: la **importancia de variables**. Existen dos enfoques alternativos con propiedades estadísticas distintas:

**1. MDI — *Mean Decrease in Impurity***  
Suma de las reducciones de impureza atribuibles a cada variable, promediadas sobre todos los árboles. Se obtiene gratuitamente durante el entrenamiento (`feature_importances_`). Pero adolece de **sesgo hacia variables con alta cardinalidad** (Strobl et al., 2007).

**2. Permutation Importance**  
Caída de la métrica de validación al permutar aleatoriamente los valores de una variable. Más costosa (requiere $R \times p$ predicciones) pero **insesgada frente a cardinalidad**. Calculada sobre conjunto de validación independiente.

In [ ]:
# Importancia MDI (gratuita desde el entrenamiento)
mdi_df = pd.DataFrame({
    'feature': X_train.columns,
    'mdi': rf.feature_importances_
}).sort_values('mdi', ascending=False).reset_index(drop=True)

print("📊 Top 15 variables por MDI (Mean Decrease in Impurity):\n")
print(mdi_df.head(15).to_string(index=False))

In [ ]:
# Importancia por permutación (sobre conjunto de TEST)
print("⏳ Calculando permutation importance (n_repeats=30)...")
perm = permutation_importance(
    rf, X_test, y_test,
    n_repeats=30,
    random_state=RNG_SEED,
    n_jobs=-1,
    scoring='roc_auc'
)

perm_df = pd.DataFrame({
    'feature':  X_train.columns,
    'perm_mean': perm.importances_mean,
    'perm_std':  perm.importances_std
}).sort_values('perm_mean', ascending=False).reset_index(drop=True)

print("\n📊 Top 15 variables por Permutation Importance:\n")
print(perm_df.head(15).to_string(index=False))

In [ ]:
# Visualización lado a lado de MDI y Permutation
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

top_n = 12

# MDI
mdi_top = mdi_df.head(top_n).iloc[::-1]
axes[0].barh(mdi_top['feature'], mdi_top['mdi'],
             color=COL_EMERALD, edgecolor=COL_NAVY, linewidth=0.5)
axes[0].set_xlabel('MDI — Reducción de impureza', fontsize=11, color=COL_NAVY)
axes[0].set_title('Top 12 variables — MDI', fontsize=12,
                  fontweight='bold', color=COL_NAVY)
axes[0].grid(axis='x', alpha=0.4)

# Permutation
perm_top = perm_df.head(top_n).iloc[::-1]
axes[1].barh(perm_top['feature'], perm_top['perm_mean'],
             xerr=perm_top['perm_std'],
             color=COL_AMBER, edgecolor=COL_NAVY, linewidth=0.5,
             error_kw={'ecolor': COL_CARMIN, 'capsize': 3, 'alpha': 0.7})
axes[1].set_xlabel('Permutation Importance — Caída de AUC', fontsize=11, color=COL_NAVY)
axes[1].set_title('Top 12 variables — Permutation', fontsize=12,
                  fontweight='bold', color=COL_NAVY)
axes[1].grid(axis='x', alpha=0.4)

plt.suptitle('Importancia de variables: MDI vs Permutation Importance',
             fontsize=14, fontweight='bold', color=COL_NAVY, y=1.02)
plt.tight_layout()
plt.show()

### 7.1. Comparación de los dos rankings

Cuando ambas medidas coinciden en el ranking de variables principales, la conclusión es **robusta**. Cuando discrepan, suele revelar problemas de correlación entre predictoras o sesgos por cardinalidad. Calculamos la correlación de Spearman entre los rankings.

In [ ]:
# Ranking comparado entre MDI y Permutation
from scipy.stats import spearmanr

mdi_rank  = mdi_df['feature'].reset_index().rename(columns={'index': 'rank_mdi'})
perm_rank = perm_df['feature'].reset_index().rename(columns={'index': 'rank_perm'})

ranking_comparado = mdi_rank.merge(perm_rank, on='feature')
ranking_comparado['diff'] = ranking_comparado['rank_mdi'] - ranking_comparado['rank_perm']

# Correlación de Spearman entre rankings
rho_spearman = spearmanr(ranking_comparado['rank_mdi'],
                         ranking_comparado['rank_perm']).correlation

print(f"🔍 Correlación de Spearman entre rankings MDI y Permutation: {rho_spearman:.4f}")
print(f"   {'(rankings prácticamente equivalentes)' if rho_spearman > 0.85 else '(rankings con discrepancias relevantes)'}\n")

print("🔝 Top 10 por MDI y posición en Permutation:")
print(ranking_comparado.head(10).to_string(index=False))

## 8. Comparativa final: CART, Random Forest y Regresión Logística

Para cerrar el arco de la Unidad 3, comparamos los tres modelos sobre las tres métricas estándar de scoring crediticio:

- **AUC-ROC**: capacidad de discriminación general
- **KS**: máxima separación entre distribuciones de buenos y malos
- **Gini**: $2 \cdot \text{AUC} - 1$, otra forma de expresar la discriminación

Para la regresión logística entrenamos una versión rápida sobre los mismos datos preprocesados, a modo de referencia. En clase 5 vimos el modelo con interpretación completa de odds ratios.

In [ ]:
# Función auxiliar para calcular KS
def calcular_ks(y_true, y_proba):
    '''Calcula el estadístico KS entre las distribuciones de score por clase.'''
    return ks_2samp(y_proba[y_true == 1], y_proba[y_true == 0]).statistic


# Entrenamiento rápido de logística como referencia
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Para la logística estandarizamos las columnas numéricas (con la convención del curso)
scaler_log = StandardScaler()
X_train_log = X_train.copy()
X_test_log  = X_test.copy()
X_train_log[num_cols] = scaler_log.fit_transform(X_train[num_cols])
X_test_log[num_cols]  = scaler_log.transform(X_test[num_cols])

logit = LogisticRegression(max_iter=5000, C=1.0, random_state=RNG_SEED, class_weight='balanced')
logit.fit(X_train_log, y_train)
p_test_log = logit.predict_proba(X_test_log)[:, 1]

# Métricas
def calcular_metricas(y_true, y_proba, nombre):
    auc = roc_auc_score(y_true, y_proba)
    ks  = calcular_ks(y_true, y_proba)
    gini = 2 * auc - 1
    return {'modelo': nombre, 'AUC': auc, 'KS': ks, 'Gini': gini}

metricas_finales = pd.DataFrame([
    calcular_metricas(y_test, p_test_log, 'Reg. Logística'),
    calcular_metricas(y_test, p_test_cart, 'Árbol CART'),
    calcular_metricas(y_test, p_test_rf,   'Random Forest'),
])

print("📊 Comparativa de modelos sobre German Credit (test set):\n")
print(metricas_finales.round(4).to_string(index=False))

In [ ]:
# Visualización: curvas ROC superpuestas
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

for prob, color, nombre in [
    (p_test_log,  COL_NAVY,    'Reg. Logística'),
    (p_test_cart, COL_VIOLET,  'Árbol CART'),
    (p_test_rf,   COL_AMBER,   'Random Forest'),
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, linewidth=2.5, color=color,
            label=f'{nombre} (AUC = {auc_val:.3f})')

# Diagonal del clasificador aleatorio
ax.plot([0, 1], [0, 1], linestyle='--', linewidth=1.5,
        color='gray', alpha=0.7, label='Clasificador aleatorio')

ax.set_xlabel('Tasa de falsos positivos (1 - Especificidad)', fontsize=12, color=COL_NAVY)
ax.set_ylabel('Tasa de verdaderos positivos (Sensibilidad)', fontsize=12, color=COL_NAVY)
ax.set_title('Curvas ROC — Tres modelos sobre German Credit',
             fontsize=13, fontweight='bold', color=COL_NAVY)
ax.legend(loc='lower right', frameon=True, fontsize=11)
ax.grid(alpha=0.4)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## 9. Síntesis y entregable

### 📊 Resumen ejecutivo

Cerramos la sesión con la tabla síntesis de los tres modelos y las conclusiones operativas del análisis. La elección final de modelo en un contexto bancario real no se reduce a la métrica de AUC: pesa también la interpretabilidad regulatoria, el costo computacional, la robustez ante deriva temporal y la facilidad de auditoría individual.

In [ ]:
# Tabla síntesis enriquecida
sintesis = pd.DataFrame({
    'Criterio':            ['AUC-ROC', 'KS', 'Gini', 'Interpretabilidad',
                            'Robustez outliers', 'Manejo categóricas',
                            'Tiempo entrenamiento', 'Aceptación regulatoria'],
    'Reg. Logística':      [f"{metricas_finales.loc[0, 'AUC']:.4f}",
                            f"{metricas_finales.loc[0, 'KS']:.4f}",
                            f"{metricas_finales.loc[0, 'Gini']:.4f}",
                            'Alta (odds ratios)',
                            'Baja',
                            'Requiere encoding',
                            'Bajo',
                            'Estándar de oro'],
    'Árbol CART':          [f"{metricas_finales.loc[1, 'AUC']:.4f}",
                            f"{metricas_finales.loc[1, 'KS']:.4f}",
                            f"{metricas_finales.loc[1, 'Gini']:.4f}",
                            'Muy alta (visual)',
                            'Alta',
                            'Nativo (incl. alta card.)',
                            'Muy bajo',
                            'Aceptable'],
    'Random Forest':       [f"{metricas_finales.loc[2, 'AUC']:.4f}",
                            f"{metricas_finales.loc[2, 'KS']:.4f}",
                            f"{metricas_finales.loc[2, 'Gini']:.4f}",
                            'Media (importancia, SHAP)',
                            'Muy alta',
                            'Nativo (incl. alta card.)',
                            'Medio (paralelizable)',
                            'Con SHAP/justificación'],
})

print("📋 Síntesis comparativa final:\n")
print(sintesis.to_string(index=False))

### 📝 Lecturas clave del análisis

**1. Performance predictiva**: Random Forest supera consistentemente a CART aislado y, en datos tabulares de tamaño moderado como el German Credit, suele superar también a la regresión logística por unos puntos de AUC. Esta ganancia, en banca, se traduce directamente en menor pérdida esperada por una mejor segmentación de cartera.

**2. Compromiso fundamental**: la elección entre los tres modelos no se reduce a la métrica de AUC. La regresión logística conserva la ventaja de la **interpretabilidad regulatoria transparente** mediante coeficientes y odds ratios. CART ofrece **reglas auditables linealmente**. Random Forest entrega **mejor performance** a costa de requerir explicabilidad complementaria (SHAP, permutation, partial dependence plots).

**3. Importancia de variables**: la coincidencia entre MDI y Permutation Importance en el ranking de variables top sugiere **robustez de las conclusiones**. Cuando aparecen discrepancias, conviene investigar correlaciones entre predictoras y posibles sesgos por cardinalidad.

**4. Sobreajuste y poda**: la curva en U del error de validación confirmó empíricamente la patología característica del árbol individual. La selección de `ccp_alpha` por validación cruzada produjo un árbol final con mejor brecha train-test que el árbol inicial sin poda dirigida.

---

### 📋 Entregable para la Clase 7

Sobre la base de este notebook, completen las siguientes consignas para entregar al inicio de la Clase 7 (jueves 21 de mayo):

1. **Replicar el pipeline completo** sobre el German Credit (sin modificaciones).
2. **Reportar las 10 variables más importantes** según MDI y según Permutation Importance, presentando ambos rankings lado a lado.
3. **Interpretar económicamente las 3 variables más predictivas**: ¿por qué tiene sentido financiero que sean las top predictoras del default?
4. **Comparar el Random Forest contra la regresión logística** de la Clase 5: ¿la mejora de AUC justificaría el cambio de modelo en producción? ¿qué consideraciones regulatorias agregarían al análisis?
5. **Documentar las decisiones** en celdas markdown intercaladas: cada elección de hiperparámetro debe estar justificada.

**Formato**: notebook `.ipynb` subido al campus virtual antes del jueves 21 de mayo, 18:00 hs.

---

### 🌳 Próxima clase — Clase 7 (21 de mayo)

- **Gradient Boosting**: aprendizaje secuencial sobre residuos
- **XGBoost**: regularización, sparsity-aware split, tree pruning
- **SVM**: hiperplano de margen máximo, *kernel trick*
- **Redes neuronales**: MLP, activaciones, backpropagation
- **Calibración**: grid search, random search, optimización bayesiana

> 📚 Lectura sugerida: Apunte de Cátedra, Unidad 3, secciones 3.2 y 3.3.  
> Hastie, Tibshirani & Friedman (2009), *The Elements of Statistical Learning*, capítulos 9 y 15.
